# Seaquest actor — single NORMALIZED lambda=0.5 run (checkpointed, resumable)
Runs ONLY the gradient-normalized (`--balance-grad`) `lambda=0.5` actor, same config as the successful run (full-view oracle critic, fp32 policy, 20k steps, ent_coef=0.01, per-state z-scored critic term, gradient-norm-balanced so lambda truly interpolates).

**Checkpoints go to Google Drive**, not just `/content`. Saves `best_val_actor.pt`, `final_actor.pt`, `latest_resume.pt`. Supports **resume** from `latest_resume.pt`. After training: reload with `strict=True`, one validation pass with **finite-logit** check, recompute offline metrics. Prints `RUN_COMPLETE` **only if** the Drive checkpoints exist AND the strict reload + finite-logit validation pass.

Does NOT run the lambda sweep and does NOT start closed-loop evaluation. **Needs:** GitHub TOKEN + Drive `raw_hf.zip` + Drive `critic_full_view.pt`.

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')

In [ ]:
# 1. Clone repo (modules only; the training loop lives in this notebook).
TOKEN = 'PASTE_YOUR_GITHUB_TOKEN_HERE'
import os, subprocess
D = '/content/Goal-Conditioned-Confounded-MsPacman'
if not os.path.isdir(D):
    subprocess.run(['git','clone',f'https://{TOKEN}@github.com/tingrui-huang/Goal-Conditioned-Confounded-MsPacman.git',D], check=True)
%cd /content/Goal-Conditioned-Confounded-MsPacman
!git pull -q && git log --oneline -1

In [ ]:
# 2. Mount Drive; load raw_hf (DATA_ROOT) and set the Drive checkpoint dir.
import glob, os, zipfile
from google.colab import drive
drive.mount('/content/drive')
ZIP = '/content/drive/MyDrive/raw_hf.zip'                     # <-- EDIT if elsewhere
assert os.path.exists(ZIP), f'zip not found at {ZIP}'
EXTRACT = '/content/raw_hf_extracted'
if not glob.glob(EXTRACT + '/**/traj_0000.npz', recursive=True):
    with zipfile.ZipFile(ZIP) as z: z.extractall(EXTRACT)
DATA_ROOT = os.path.dirname(glob.glob(EXTRACT + '/**/traj_0000.npz', recursive=True)[0])
assert len(glob.glob(DATA_ROOT + '/traj_*.npz')) == 40
DRIVE_OUT = '/content/drive/MyDrive/seaquest_actor_lam0.5_normalized'   # checkpoints -> DRIVE
os.makedirs(DRIVE_OUT, exist_ok=True)
print('DATA_ROOT =', DATA_ROOT, '| DRIVE_OUT =', DRIVE_OUT)

In [ ]:
# 3. Locate the PRE-TRAINED full-view (oracle) critic on Drive.
import glob
CRITIC = '/content/drive/MyDrive/critic_full_view.pt'        # <-- EDIT if elsewhere
if not os.path.exists(CRITIC):
    hits = glob.glob('/content/drive/MyDrive/**/critic_full_view.pt', recursive=True)
    assert hits, 'critic_full_view.pt not found on Drive'
    CRITIC = hits[0]
c = torch.load(CRITIC, map_location='cpu', weights_only=False)
print('CRITIC =', CRITIC, '| oracle =', c['oracle'], '| frame_stack =', c['cfg'].get('frame_stack'))
assert c['oracle'] is True and c['cfg'].get('frame_stack') == 4

In [ ]:
# 4. CONFIG (same as the successful run) — ONLY lambda=0.5, gradient-NORMALIZED.
LAM = 0.5
BALANCE_GRAD = True          # gradient-norm-balanced => lambda genuinely interpolates BC<->critic
STEPS = 20000
ENT_COEF = 0.01
CKPT_EVERY = 1000            # latest_resume.pt cadence
VAL_EVERY = 1000             # validation + best_val_actor.pt cadence
VAL_N = 4096                 # held-out validation batch size
print(dict(lam=LAM, balance_grad=BALANCE_GRAD, steps=STEPS, ent_coef=ENT_COEF))

In [ ]:
# 5. Build critic(frozen)/actor/optimizer + train & held-out validation samplers; RESUME if present.
import numpy as np, torch.nn.functional as F, json, time
from seaquest_ccrl.training.train_critic import load_critic
from seaquest_ccrl.training.dataset_sampler import HindsightSampler
from seaquest_ccrl.models.actor import GoalConditionedActor
from seaquest_ccrl.games import get_game
device = 'cuda' if torch.cuda.is_available() else 'cpu'
critic, cfg, oracle = load_critic(CRITIC, device)
assert oracle is True and cfg.frame_stack == 4
for p in critic.parameters(): p.requires_grad_(False)
critic.eval()
game = get_game('seaquest'); A = cfg.nb_actions
train_S = HindsightSampler(game, oracle=True, cfg=cfg, device=device, rng=np.random.default_rng(cfg.seed + 777), root=DATA_ROOT)
val_S   = HindsightSampler(game, oracle=True, cfg=cfg, device=device, rng=np.random.default_rng(cfg.seed + 999), root=DATA_ROOT)
VFR, VA0, VGO = val_S.sample(VAL_N)                          # FIXED held-out validation batch
actor = GoalConditionedActor(cfg.frame_size, A, cfg.frame_stack).to(device)
opt = torch.optim.Adam(actor.parameters(), lr=cfg.lr)
use_amp = (device == 'cuda')
start_step, best_bcacc = 1, -1.0
RESUME = os.path.join(DRIVE_OUT, 'latest_resume.pt')
if os.path.exists(RESUME):
    ck = torch.load(RESUME, map_location=device)
    actor.load_state_dict(ck['actor']); opt.load_state_dict(ck['opt'])
    start_step = int(ck['step']) + 1; best_bcacc = float(ck.get('best_bcacc', -1.0))
    print(f'RESUMED from {RESUME} @ step {start_step} (best_val_BCacc={best_bcacc:.3f})')
else:
    print('fresh start (no latest_resume.pt in Drive)')

In [ ]:
# 6. TRAIN (faithful objective: (1-lam)*Epi[zscore(f)] + lam*BC + ent*H, fp32 policy, grad-balanced).
def actor_loss(frames, a_orig, goals):
    with torch.no_grad(), torch.autocast(device_type='cuda' if use_amp else 'cpu', dtype=torch.float16, enabled=use_amp):
        g_emb = critic.g_encoder(goals)
        f_all = torch.stack([(critic.sa_encoder(frames, torch.full((frames.shape[0],), a, device=device, dtype=torch.long)) * g_emb).sum(1) for a in range(A)], 1).float()
    logits = actor(frames, goals); logp = F.log_softmax(logits, 1); pi = logp.exp()       # fp32 policy
    f_c = f_all - f_all.mean(1, keepdim=True); f_n = f_c / f_c.std(1, keepdim=True).clamp_min(1e-6)
    critic_term = (pi * f_n).sum(1).mean()
    bc_term = logp.gather(1, a_orig.view(-1, 1)).squeeze(1).mean()
    ent_term = -(pi * logp).sum(1).mean()
    if BALANCE_GRAD:
        params = [p for p in actor.parameters() if p.requires_grad]
        gc = torch.autograd.grad(critic_term, params, retain_graph=True, allow_unused=True)
        gb = torch.autograd.grad(bc_term, params, retain_graph=True, allow_unused=True)
        nc = torch.sqrt(sum((g.detach()**2).sum() for g in gc if g is not None)).clamp_min(1e-8)
        nb = torch.sqrt(sum((g.detach()**2).sum() for g in gb if g is not None))
        critic_term = critic_term * (nb / nc).detach()
    return -((1.0 - LAM) * critic_term + LAM * bc_term + ENT_COEF * ent_term)

@torch.no_grad()
def validate():
    actor.eval(); lg = actor(VFR, VGO); actor.train()
    bcacc = float((lg.argmax(1) == VA0).float().mean())
    return bcacc, bool(torch.isfinite(lg).all())

def save_resume(step):
    torch.save({'actor': actor.state_dict(), 'opt': opt.state_dict(), 'step': step, 'best_bcacc': best_bcacc,
                'cfg': cfg.__dict__, 'lam': LAM, 'ent_coef': ENT_COEF, 'balance_grad': BALANCE_GRAD},
               os.path.join(DRIVE_OUT, 'latest_resume.pt'))

actor.train(); t0 = time.time()
for step in range(start_step, STEPS + 1):
    fr, a0, go = train_S.sample(cfg.batch_size)
    loss = actor_loss(fr, a0, go)
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    if step % VAL_EVERY == 0:
        bcacc, finite = validate()
        if bcacc > best_bcacc:
            best_bcacc = bcacc
            torch.save({'state_dict': actor.state_dict(), 'cfg': cfg.__dict__, 'lam': LAM, 'step': step,
                        'val_bc_acc': bcacc, 'oracle': True, 'balance_grad': BALANCE_GRAD},
                       os.path.join(DRIVE_OUT, 'best_val_actor.pt'))
        print(f'step {step:6d}/{STEPS}  loss {float(loss):+.4f}  val_BCacc {bcacc:.3f}  best {best_bcacc:.3f}  finite={finite}  {step/(time.time()-t0):.1f} it/s')
    if step % CKPT_EVERY == 0:
        save_resume(step)
torch.save({'state_dict': actor.state_dict(), 'cfg': cfg.__dict__, 'lam': LAM, 'step': STEPS,
            'oracle': True, 'balance_grad': BALANCE_GRAD}, os.path.join(DRIVE_OUT, 'final_actor.pt'))
save_resume(STEPS)
print('saved -> best_val_actor.pt / final_actor.pt / latest_resume.pt @', DRIVE_OUT)

In [ ]:
# 7. Reload (strict=True) + finite-logit validation + recompute offline metrics; gate RUN_COMPLETE.
REQUIRED = ['best_val_actor.pt', 'final_actor.pt', 'latest_resume.pt']
drive_ok = all(os.path.exists(os.path.join(DRIVE_OUT, f)) for f in REQUIRED)
ckpt_path = os.path.join(DRIVE_OUT, 'best_val_actor.pt' if os.path.exists(os.path.join(DRIVE_OUT, 'best_val_actor.pt')) else 'final_actor.pt')
reload_ok = finite_ok = False; metrics = None
try:
    ck = torch.load(ckpt_path, map_location=device)
    ra = GoalConditionedActor(cfg.frame_size, A, cfg.frame_stack).to(device)
    ra.load_state_dict(ck['state_dict'], strict=True)                       # STRICT reload
    ra.eval(); reload_ok = True
    with torch.no_grad():
        lg = ra(VFR, VGO); finite_ok = bool(torch.isfinite(lg).all())       # one validation pass, finite logits
        pi = F.softmax(lg, 1); aa = lg.argmax(1)
        ge = critic.g_encoder(VGO)
        fall = torch.stack([(critic.sa_encoder(VFR, torch.full((len(VFR),), a, device=device, dtype=torch.long)) * ge).sum(1) for a in range(A)], 1)
        ac = fall.argmax(1)
        metrics = {'top1_agree_teacher': float((aa == VA0).float().mean()),
                   'top1_agree_argmax_critic': float((aa == ac).float().mean()),
                   'actor_entropy_nats': float(-(pi * torch.log(pi.clamp_min(1e-9))).sum(1).mean()),
                   'critic_value_actor_Epi_f': float((pi * fall).sum(1).mean()),
                   'critic_value_argmax_f': float(fall.max(1).values.mean()),
                   'val_bc_acc': float((aa == VA0).float().mean())}
    json.dump({'checkpoint': ckpt_path, 'lam': LAM, 'balance_grad': BALANCE_GRAD,
               'drive_files_ok': drive_ok, 'reload_strict_ok': reload_ok, 'finite_logits': finite_ok,
               'metrics': metrics}, open(os.path.join(DRIVE_OUT, 'offline_metrics.json'), 'w'), indent=2)
except Exception as e:
    print('RELOAD/VALIDATION FAILED:', type(e).__name__, e)

print('drive_files_ok =', drive_ok, '| reload_strict_ok =', reload_ok, '| finite_logits =', finite_ok)
print('offline metrics:', json.dumps(metrics, indent=2) if metrics else None)
if drive_ok and reload_ok and finite_ok:
    print('RUN_COMPLETE')
else:
    print('NOT COMPLETE - Drive checkpoint missing or reload/finite-logit validation failed; RUN_COMPLETE withheld')